# Pelatihan pengklasifikasi TrashScan — BinGo

Melatih pengklasifikasi kemasan dari tiga dataset publik berlisensi jelas, lalu
melaporkan angkanya dengan cara yang tahan ditanyai juri.

**Baca ini dulu.** Dataset publik hanya mencapai **lapis material** (8 kelas).
Papan harga BinGo bekerja pada **lapis grade** (18 kelas), dan selisih harga
terbesar justru ada pada pembedaan yang tidak dilabeli dataset mana pun — bening
versus berwarna, koran versus duplex, tembaga versus besi. Dari 18 grade, hanya
3 yang bisa diturunkan dari data publik tanpa menebak.

Artinya model ini **asisten identifikasi kasar**, bukan penentu harga. Kode resin
tetap jalur utama karena ia fakta, bukan tebakan. Kalimat itu akan ikut tercetak
di laporan akhir supaya tidak hilang saat naskah disalin.

Yang dilakukan notebook ini:

1. Unduh dan satukan tiga dataset (Drinking Waste CC0, TrashNet MIT, RealWaste CC BY 4.0)
2. Buang duplikat lintas-sumber dengan perceptual hash
3. Pisahkan train/val/test **per klaster foto**, bukan per foto
4. Transfer learning MobileNetV3-Small
5. Evaluasi: macro-F1, per kelas, confusion matrix
6. Kalibrasi keyakinan dengan temperature scaling
7. Kurva abstain — dari sinilah ambang di aplikasi ditetapkan
8. **Uji lintas-dataset** — angka paling jujur soal daya tahan di lapangan
9. Ekspor TFLite int8 beserta ukuran dan latensi

Jalankan di Colab dengan GPU (Runtime → Change runtime type → T4). Sekitar 20–30
menit untuk seluruh notebook.

In [ ]:
# @title Setup
SMOKE = False   # True = pakai citra sintetis, untuk memeriksa notebook berjalan tanpa mengunduh apa pun
SEED  = 1337
IMG   = 224

import os, sys, json, random, shutil, subprocess, time, csv
from pathlib import Path
from collections import Counter, defaultdict

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q tensorflow scikit-learn pillow kaggle huggingface_hub requests

import numpy as np
random.seed(SEED); np.random.seed(SEED)
import tensorflow as tf
tf.random.set_seed(SEED)
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU') or 'tidak ada')

In [ ]:
# @title Ambil berkas pendukung dari repositori
# label_map.py dan prepare_dataset.py berada di tools/trashscan-model/.
HERE = Path.cwd()
if not (HERE / 'label_map.py').exists():
    for cand in [HERE.parent, HERE / 'tools' / 'trashscan-model',
                 Path('/content/BinGo/tools/trashscan-model')]:
        if (cand / 'label_map.py').exists():
            os.chdir(cand); break
    else:
        if IN_COLAB:
            raise SystemExit('Unggah label_map.py dan prepare_dataset.py, atau clone repo dulu:\n'
                             '  !git clone <url-repo> /content/BinGo')
sys.path.insert(0, str(Path.cwd()))
from label_map import (app_mapping_table, TRAIN_CLASS_TO_APP,
                       UNREACHABLE_MATERIAL_TYPES,
                       MATERIALS, MATERIAL_LABEL_ID, SOURCE_META, REACHABLE_GRADES,
                       ALL_GRADES, coverage_report)
print(coverage_report())

In [ ]:
# @title Gaya grafik
# Warna kategorikal dipakai dalam urutan tetap dan sudah lolos pemeriksaan
# keterbacaan bagi pembaca dengan buta warna (CVD ΔE 9.2, normal 24.0, all-pairs).
import matplotlib as mpl
import matplotlib.pyplot as plt

SURFACE   = '#fcfcfb'
INK       = '#0b0b0b'
INK_MUTED = '#52514e'
SERIES    = ['#2a78d6', '#eb6834', '#1baf7a']          # biru, oranye, aqua
SEQ       = ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#2a78d6', '#256abf', '#1c5cab', '#184f95']
SEQ_CMAP  = mpl.colors.LinearSegmentedColormap.from_list('bingo_seq', SEQ)

mpl.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'text.color': INK, 'axes.labelcolor': INK_MUTED, 'axes.titlecolor': INK,
    'xtick.color': INK_MUTED, 'ytick.color': INK_MUTED,
    'axes.edgecolor': '#d9d8d4', 'grid.color': '#ebeae6', 'grid.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': '600',
    'figure.dpi': 120, 'lines.linewidth': 2, 'legend.frameon': False,
})
print('siap')

## 1 — Dataset

Tiga sumber, dipilih karena lisensinya jelas dan saling menutupi kelemahan.
TrashNet difoto di atas posterboard putih; model yang hanya belajar dari situ
ambruk pada foto ponsel sungguhan. RealWaste difoto di titik penerimaan TPA.
Keduanya dipakai bersama, lalu diuji silang di langkah 7.

Drinking Waste perlu akun Kaggle. Token diambil dari kaggle.com → Settings → API
→ Create New Token, lalu unggah `kaggle.json` di sel berikut bila di Colab.

In [ ]:
# @title Unduh dan satukan
DATA = Path('data/unified')

if SMOKE:
    from PIL import Image
    rng = np.random.default_rng(SEED)
    # Warna diambil dari MATERIALS supaya mode uji ikut berubah sendiri ketika
    # daftar kelas berubah. Versi sebelumnya mengetik nama kelas secara manual,
    # dan begitu taksonominya diganti, seluruh manifest sintetis menjadi kosong
    # tanpa satu pun galat sampai tiga sel kemudian.
    base = [(150,190,225), (225,228,230), (120,175,150), (228,225,214), (172,132,92),
            (196,168,120), (168,170,175), (172,202,206), (120,158,104), (108,100,94)]
    tint = {m: base[i % len(base)] for i, m in enumerate(MATERIALS)}
    rows = []
    for mi, (m, t) in enumerate(tint.items()):
        d = DATA / 'images' / m; d.mkdir(parents=True, exist_ok=True)
        for i in range(48):
            src = ['trashnet', 'realwaste', 'drinking_waste'][i % 3]
            a = np.clip(rng.normal(t, 24, (72, 72, 3)), 0, 255).astype(np.uint8)
            p = d / f'{src}__{m}__{i}.jpg'
            Image.fromarray(a).save(p, quality=88)
            rows.append({'path': str(p.relative_to(DATA)), 'source': src, 'raw_class': m,
                         'material': m, 'grade': '', 'cluster': mi * 1000 + i,
                         'dhash': 0, 'note': ''})
    with (DATA / 'manifest.csv').open('w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    print(f'mode uji: {len(rows)} citra sintetis, {len(tint)} kelas')
else:
    if IN_COLAB and not Path.home().joinpath('.kaggle/kaggle.json').exists():
        from google.colab import files
        print('Unggah kaggle.json (batal saja bila ingin melewati Drinking Waste):')
        try:
            up = files.upload()
            Path.home().joinpath('.kaggle').mkdir(exist_ok=True)
            Path.home().joinpath('.kaggle/kaggle.json').write_bytes(list(up.values())[0])
            os.chmod(Path.home() / '.kaggle/kaggle.json', 0o600)
        except Exception as e:
            print('dilewati:', e)
    srcs = ['trashnet', 'realwaste']
    if Path.home().joinpath('.kaggle/kaggle.json').exists():
        srcs.insert(0, 'drinking_waste')
    subprocess.run([sys.executable, 'prepare_dataset.py', '--out', str(DATA),
                    '--sources', *srcs], check=True)

man = list(csv.DictReader((DATA / 'manifest.csv').open()))
print(f'\n{len(man):,} citra')
print('per sumber :', dict(Counter(r['source'] for r in man)))
print('per material:', dict(Counter(r['material'] for r in man)))

In [ ]:
# @title Sebaran kelas
# Satu seri, jadi satu warna dan tanpa legenda — judulnya sudah menyebut apa yang diukur.
counts = Counter(r['material'] for r in man)
order  = [m for m in MATERIALS if counts.get(m)]
vals   = [counts[m] for m in order]

fig, ax = plt.subplots(figsize=(7, 0.42 * len(order) + 1.4))
y = np.arange(len(order))
ax.barh(y, vals, height=0.62, color=SERIES[0], zorder=3)
ax.set_yticks(y, [MATERIAL_LABEL_ID[m] for m in order])
ax.invert_yaxis()
ax.xaxis.grid(True, zorder=0); ax.set_axisbelow(True)
ax.set_xlabel('jumlah citra')
ax.set_title('Sebaran citra per material')
for yi, v in zip(y, vals):
    ax.text(v + max(vals) * 0.012, yi, f'{v:,}', va='center', color=INK_MUTED, fontsize=9)
ax.set_xlim(0, max(vals) * 1.12)
plt.tight_layout(); plt.show()

imb = max(vals) / min(vals)
print(f'Rasio ketimpangan terbesar : terkecil = {imb:.1f}x')
print('Karena itu metrik utamanya macro-F1, bukan akurasi — akurasi menyembunyikan kelas kecil.')

In [ ]:
# @title Contoh citra per kelas
by_mat = defaultdict(list)
for r in man:
    by_mat[r['material']].append(r)
ncol = 6
fig, axes = plt.subplots(len(order), ncol, figsize=(ncol * 1.35, len(order) * 1.45))
if len(order) == 1:
    axes = np.array([axes])
from PIL import Image
for i, m in enumerate(order):
    sample = random.Random(SEED).sample(by_mat[m], min(ncol, len(by_mat[m])))
    for j in range(ncol):
        ax = axes[i, j]; ax.axis('off')
        if j < len(sample):
            ax.imshow(Image.open(DATA / sample[j]['path']).convert('RGB').resize((96, 96)))
        if j == 0:
            ax.set_title(MATERIAL_LABEL_ID[m], loc='left', fontsize=9, color=INK)
plt.tight_layout(); plt.show()

## 2 — Pemisahan train/val/test

Dipisah **per klaster foto**, bukan per foto. `prepare_dataset.py` sudah
mengelompokkan citra yang nyaris identik dengan difference hash; di sini
klaster itu yang dibagi, sehingga dua foto objek fisik yang sama tidak pernah
jatuh di train dan test sekaligus.

Ini kesalahan yang paling sering menaikkan akurasi secara semu pada laporan
klasifikasi sampah. Kalau angkamu nanti lebih rendah dari makalah yang kamu
baca, kemungkinan besar karena kamu memisahkan dengan benar dan mereka tidak.

Rasio 70/13/17 mengikuti Thung & Yang (2016) supaya angkanya bisa disandingkan
langsung dengan 75% yang mereka laporkan.

In [ ]:
# @title Split per klaster, proporsi kelas dijaga
def split_by_cluster(rows, ratios=(0.70, 0.13, 0.17), seed=SEED):
    rng = random.Random(seed)
    groups = defaultdict(list)
    for i, r in enumerate(rows):
        groups[r['cluster']].append(i)
    by_class = defaultdict(list)
    for cid, idxs in groups.items():
        major = Counter(rows[i]['material'] for i in idxs).most_common(1)[0][0]
        by_class[major].append(idxs)
    tr, va, te = [], [], []
    for cls, gs in by_class.items():
        rng.shuffle(gs)
        n_tr = int(round(len(gs) * ratios[0])); n_va = int(round(len(gs) * ratios[1]))
        for g in gs[:n_tr]: tr += g
        for g in gs[n_tr:n_tr + n_va]: va += g
        for g in gs[n_tr + n_va:]: te += g
    for s in (tr, va, te): rng.shuffle(s)
    return tr, va, te

CLASSES = order
tr_i, va_i, te_i = split_by_cluster(man)
print(f'train {len(tr_i):,} | val {len(va_i):,} | test {len(te_i):,}')
print('klaster unik:', len({r["cluster"] for r in man}), 'dari', len(man), 'foto')
for name, idx in [('train', tr_i), ('val', va_i), ('test', te_i)]:
    c = Counter(man[i]['material'] for i in idx)
    print(f'  {name:5s}', {m: c.get(m, 0) for m in CLASSES})

In [ ]:
# @title Pipeline data
AUTOTUNE = tf.data.AUTOTUNE
BATCH = 32

def make_ds(idxs, training):
    paths = [str(DATA / man[i]['path']) for i in idxs]
    ys = [CLASSES.index(man[i]['material']) for i in idxs]
    def load(p, y):
        img = tf.image.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
        return tf.cast(tf.image.resize(img, (IMG, IMG)), tf.float32), y
    ds = tf.data.Dataset.from_tensor_slices((paths, ys))
    if training:
        ds = ds.shuffle(min(len(paths), 2048), seed=SEED)
    ds = ds.map(load, num_parallel_calls=AUTOTUNE)
    if training:
        def aug(x, y):
            x = tf.image.random_flip_left_right(x)
            x = tf.image.random_brightness(x, 0.15)
            x = tf.image.random_contrast(x, 0.85, 1.15)
            return tf.clip_by_value(x, 0., 255.), y
        ds = ds.map(aug, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

ds_tr, ds_va, ds_te = make_ds(tr_i, True), make_ds(va_i, False), make_ds(te_i, False)
print('siap')

## 3 — Model

MobileNetV3-Small dengan bobot ImageNet. Dipilih karena harus muat dan cepat di
ponsel kelas bawah, bukan karena ia arsitektur terbaik — dan itu memang bukan.

Dua tahap: kepala dilatih dulu dengan backbone dibekukan, lalu 40 lapis terakhir
dibuka pada laju belajar kecil. Bobot kelas dipakai untuk menangani ketimpangan
jumlah.

In [ ]:
# @title Latih
EPOCHS_HEAD, EPOCHS_FT = (1, 1) if SMOKE else (10, 8)

backbone = tf.keras.applications.MobileNetV3Small(
    input_shape=(IMG, IMG, 3), include_top=False,
    weights=None if SMOKE else 'imagenet', include_preprocessing=True, pooling='avg')
backbone.trainable = False
inp = tf.keras.Input((IMG, IMG, 3))
x = tf.keras.layers.Dropout(0.2)(backbone(inp, training=False))
model = tf.keras.Model(inp, tf.keras.layers.Dense(len(CLASSES), name='logits')(x))

ytr = np.array([CLASSES.index(man[i]['material']) for i in tr_i])
cnt = Counter(ytr.tolist())
cw = {c: len(ytr) / (len(CLASSES) * cnt.get(c, 1)) for c in range(len(CLASSES))}
print('bobot kelas:', {CLASSES[c]: round(v, 2) for c, v in cw.items()})

loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(tf.keras.optimizers.Adam(1e-3), loss=loss, metrics=['accuracy'])
h1 = model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS_HEAD, class_weight=cw, verbose=2)

backbone.trainable = True
for l in backbone.layers[:-40]:
    l.trainable = False
model.compile(tf.keras.optimizers.Adam(1e-5), loss=loss, metrics=['accuracy'])
h2 = model.fit(ds_tr, validation_data=ds_va, epochs=EPOCHS_FT, class_weight=cw, verbose=2)

## 4 — Evaluasi

In [ ]:
# @title Macro-F1 dan tabel per kelas
def per_class_prf(yt, yp, n):
    out = {}
    for c in range(n):
        tp = int(((yp == c) & (yt == c)).sum()); fp = int(((yp == c) & (yt != c)).sum())
        fn = int(((yp != c) & (yt == c)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        out[c] = {'precision': p, 'recall': r,
                  'f1': 2 * p * r / (p + r) if p + r else 0.0, 'support': int((yt == c).sum())}
    return out

def macro_f1(yt, yp, n):
    d = per_class_prf(yt, yp, n)
    return float(np.mean([d[c]['f1'] for c in range(n)]))

def logits_of(ds):
    lg = model.predict(ds, verbose=0)
    ys = np.concatenate([y.numpy() for _, y in ds])
    return lg, ys

lg_va, y_va = logits_of(ds_va)
lg_te, y_te = logits_of(ds_te)
pred0 = lg_te.argmax(1)
N = len(CLASSES)

print(f'Akurasi  {(pred0 == y_te).mean():.3f}')
print(f'Macro-F1 {macro_f1(y_te, pred0, N):.3f}\n')
d = per_class_prf(y_te, pred0, N)
print(f'{"kelas":<40}{"P":>7}{"R":>7}{"F1":>7}{"n":>7}')
for c in range(N):
    v = d[c]
    print(f'{MATERIAL_LABEL_ID[CLASSES[c]]:<40}{v["precision"]:>7.3f}{v["recall"]:>7.3f}'
          f'{v["f1"]:>7.3f}{v["support"]:>7}')

In [ ]:
# @title Confusion matrix
# Besaran, jadi satu hue terang→gelap. Tiap sel diberi angka: itu sekaligus
# yang membuat grafik ini terbaca tanpa mengandalkan warna saja.
cm = np.zeros((N, N), int)
for t_, p_ in zip(y_te, pred0):
    cm[t_, p_] += 1
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(1.0 * N + 2.6, 1.0 * N + 2.0))
im = ax.imshow(cmn, cmap=SEQ_CMAP, vmin=0, vmax=1)
short = [MATERIAL_LABEL_ID[c].split(' (')[0] for c in CLASSES]
ax.set_xticks(range(N), short, rotation=40, ha='right')
ax.set_yticks(range(N), short)
ax.set_xlabel('prediksi'); ax.set_ylabel('sebenarnya')
ax.set_title('Confusion matrix (proporsi per baris)')
for i in range(N):
    for j in range(N):
        if cm[i, j]:
            ax.text(j, i, f'{cmn[i, j]:.2f}\n{cm[i, j]}', ha='center', va='center', fontsize=8,
                    color='#ffffff' if cmn[i, j] > 0.55 else INK)
ax.set_xticks(np.arange(-.5, N, 1), minor=True); ax.set_yticks(np.arange(-.5, N, 1), minor=True)
ax.grid(which='minor', color=SURFACE, linewidth=2); ax.tick_params(which='minor', length=0)
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03, label='proporsi')
plt.tight_layout(); plt.show()

conf = [(short[i], short[j], cm[i, j]) for i in range(N) for j in range(N) if i != j]
conf.sort(key=lambda t: -t[2])
print('Kekeliruan terbanyak:')
for a, b, v in conf[:5]:
    if v: print(f'  {a} dikira {b}: {v}')

## 5 — Kalibrasi

Akurasi menjawab "seberapa sering benar". Kalibrasi menjawab pertanyaan yang
berbeda dan justru lebih penting untuk aplikasi ini: **ketika model bilang yakin
80%, apakah ia benar 80% kali?** Kalau tidak, ambang abstain di aplikasi tidak
punya arti.

Temperature scaling memasang satu parameter pada data validation. Ia tidak
mengubah urutan prediksi, jadi akurasi tetap; yang berubah hanya sebaran
keyakinannya.

In [ ]:
# @title Temperature scaling + reliability diagram
def ece(conf, correct, bins=10):
    edges = np.linspace(0, 1, bins + 1); e = 0.0; rows = []
    for i in range(bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1])
        if m.any():
            acc, cf = float(correct[m].mean()), float(conf[m].mean())
            e += m.sum() / len(conf) * abs(acc - cf)
            rows.append((float((edges[i] + edges[i + 1]) / 2), cf, acc, int(m.sum())))
    return float(e), rows

best_t, best_nll = 1.0, np.inf
for t in np.arange(0.5, 5.01, 0.05):
    p = tf.nn.softmax(lg_va / t).numpy()
    nll = float(-np.mean(np.log(np.clip(p[np.arange(len(y_va)), y_va], 1e-12, 1))))
    if nll < best_nll: best_nll, best_t = nll, float(t)

prob = tf.nn.softmax(lg_te / best_t).numpy()
pred, cfd = prob.argmax(1), prob.max(1)
e_before, _ = ece(tf.nn.softmax(lg_te).numpy().max(1), (pred0 == y_te).astype(float))
e_after, rel = ece(cfd, (pred == y_te).astype(float))
print(f'T = {best_t:.2f} | ECE {e_before:.3f} -> {e_after:.3f}')

fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.plot([0, 1], [0, 1], '--', color=INK_MUTED, linewidth=1.2, label='kalibrasi sempurna')
if rel:
    ax.plot([r[1] for r in rel], [r[2] for r in rel], 'o-', color=SERIES[0],
            markersize=7, label=f'setelah scaling (T={best_t:.2f})')
ax.set_xlabel('keyakinan model'); ax.set_ylabel('akurasi sebenarnya')
ax.set_title('Reliability diagram'); ax.grid(True); ax.set_axisbelow(True)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

## 6 — Perilaku abstain

Aplikasi menahan jawaban ketika model tidak yakin, lalu meminta pengguna memilih
material sendiri atau memotret kode resin. Tabel ini yang menentukan ambangnya.

Dua garis, dua besaran, **satu sumbu** — keduanya berskala 0–1, jadi tidak perlu
sumbu ganda. Semakin tinggi ambang, semakin sedikit pertanyaan yang dijawab
(cakupan turun) tetapi yang dijawab makin benar. Titik yang dicari adalah tempat
macro-F1 sudah tinggi sementara cakupan belum runtuh.

In [ ]:
# @title Kurva abstain
ths = np.arange(0.0, 0.96, 0.05)
cov, f1s = [], []
for t in ths:
    k = cfd >= t
    cov.append(float(k.mean()))
    f1s.append(macro_f1(y_te[k], pred[k], N) if k.sum() else np.nan)

fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(ths, cov, color=SERIES[0], label='cakupan (dijawab)')
ax.plot(ths, f1s, color=SERIES[1], label='macro-F1 pada yang dijawab')
ax.set_xlabel('ambang keyakinan'); ax.set_ylabel('proporsi')
ax.set_title('Cakupan versus mutu jawaban'); ax.grid(True); ax.set_axisbelow(True)
ax.set_ylim(0, 1.02); ax.legend(loc='lower left')

ok = [i for i in range(len(ths)) if cov[i] >= 0.70 and not np.isnan(f1s[i])]
pick = max(ok, key=lambda i: f1s[i]) if ok else int(np.nanargmax(f1s))
TH = float(ths[pick])
ax.axvline(TH, color=INK_MUTED, linestyle=':', linewidth=1.2)
ax.text(TH + 0.01, 0.06, f'ambang {TH:.2f}', color=INK_MUTED, fontsize=9)
ax.text(ths[-1], cov[-1], f' {cov[-1]:.2f}', color=SERIES[0], fontsize=9, va='center')
ax.text(ths[-1], f1s[-1], f' {f1s[-1]:.2f}', color=SERIES[1], fontsize=9, va='center')
plt.tight_layout(); plt.show()

print(f'Ambang terpilih {TH:.2f} — cakupan {cov[pick]:.0%}, macro-F1 {f1s[pick]:.3f}')
print('Kriteria: macro-F1 tertinggi selama cakupan masih di atas 70%.')
print('Angka inilah yang dipasang di visualClassifier.ts.')

## 7 — Uji lintas-dataset

Ini bagian yang paling jujur, dan yang paling berguna dihadapkan ke juri justru
karena ia tidak memihak.

TrashNet difoto di atas posterboard putih. RealWaste difoto di titik penerimaan
TPA. Model yang belajar dari satu sumber lalu diuji pada sumber lain akan turun
angkanya — **seberapa jauh turunnya adalah perkiraan terbaik tentang apa yang
terjadi saat model bertemu foto ponsel pemulung di lapangan.**

Angka dalam-sumber yang tinggi tanpa angka lintas-sumber adalah klaim yang belum
diuji.

In [ ]:
# @title Latih per sumber, uji silang
sources = sorted({r['source'] for r in man})
res = {}
if len(sources) < 2:
    print('Butuh minimal dua sumber untuk uji silang.')
else:
    for s_tr in sources:
        idx_tr = [i for i in tr_i if man[i]['source'] == s_tr]
        if len(idx_tr) < 40:
            continue
        bb = tf.keras.applications.MobileNetV3Small(
            input_shape=(IMG, IMG, 3), include_top=False,
            weights=None if SMOKE else 'imagenet', include_preprocessing=True, pooling='avg')
        bb.trainable = False
        i_ = tf.keras.Input((IMG, IMG, 3))
        m_ = tf.keras.Model(i_, tf.keras.layers.Dense(N)(tf.keras.layers.Dropout(0.2)(bb(i_, training=False))))
        m_.compile(tf.keras.optimizers.Adam(1e-3), loss=loss, metrics=['accuracy'])
        m_.fit(make_ds(idx_tr, True), epochs=EPOCHS_HEAD, verbose=0)
        for s_te in sources:
            idx_te = [i for i in te_i if man[i]['source'] == s_te]
            if len(idx_te) < 10:
                continue
            d_ = make_ds(idx_te, False)
            p_ = m_.predict(d_, verbose=0).argmax(1)
            yt_ = np.concatenate([y.numpy() for _, y in d_])
            res[(s_tr, s_te)] = macro_f1(yt_, p_, N)
        del m_, bb

    if res:
        rows_ = sorted({a for a, _ in res}); cols_ = sorted({b for _, b in res})
        M = np.full((len(rows_), len(cols_)), np.nan)
        for (a, b), v in res.items():
            M[rows_.index(a), cols_.index(b)] = v
        fig, ax = plt.subplots(figsize=(1.5 * len(cols_) + 3.0, 1.2 * len(rows_) + 2.4))
        im = ax.imshow(M, cmap=SEQ_CMAP, vmin=0, vmax=1)
        ax.set_xticks(range(len(cols_)), cols_, rotation=20, ha='right')
        ax.set_yticks(range(len(rows_)), rows_)
        ax.set_xlabel('diuji pada'); ax.set_ylabel('dilatih pada')
        ax.set_title('Macro-F1 lintas-dataset')
        for i in range(len(rows_)):
            for j in range(len(cols_)):
                if not np.isnan(M[i, j]):
                    ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=11,
                            color='#ffffff' if M[i, j] > 0.55 else INK)
        fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03, label='macro-F1')
        plt.tight_layout(); plt.show()

        diag = [M[i, i] for i in range(min(len(rows_), len(cols_))) if not np.isnan(M[i, i])]
        off = [M[i, j] for i in range(len(rows_)) for j in range(len(cols_))
               if i != j and not np.isnan(M[i, j])]
        if diag and off:
            print(f'Rata-rata dalam-sumber {np.mean(diag):.3f} | lintas-sumber {np.mean(off):.3f}')
            print(f'Selisih {np.mean(diag) - np.mean(off):.3f} — inilah bagian yang hilang '
                  'saat model bertemu kondisi pemotretan yang belum pernah dilihatnya.')

## 8 — Ekspor ke TFLite int8

In [ ]:
# @title Kuantisasi, ukuran, latensi
OUT = Path('artifacts'); OUT.mkdir(exist_ok=True)
sm = OUT / '_saved_model'
if sm.exists(): shutil.rmtree(sm)
model.export(str(sm))

conv = tf.lite.TFLiteConverter.from_saved_model(str(sm))
conv.optimizations = [tf.lite.Optimize.DEFAULT]
rep_idx = tr_i[:200]
def rep():
    for i in rep_idx:
        img = tf.image.decode_image(tf.io.read_file(str(DATA / man[i]['path'])),
                                    channels=3, expand_animations=False)
        yield [tf.expand_dims(tf.cast(tf.image.resize(img, (IMG, IMG)), tf.float32), 0).numpy()]
conv.representative_dataset = rep
tfl = conv.convert()
(OUT / 'model_int8.tflite').write_bytes(tfl)
(OUT / 'labels.txt').write_text('\n'.join(CLASSES) + '\n')

# Jembatan ke enum aplikasi. Tanpa berkas ini, keluaran model tidak ada
# artinya bagi aplikasi: nama kelas seperti CARDBOARD dan METAL_CAN bukan
# nilai MaterialType yang sah, sehingga tidak bisa masuk ScanResult,
# gradesForMaterial(), penyaring titik setor, maupun papan harga.
(OUT / 'app_labels.json').write_text(json.dumps({
    'trainClasses': CLASSES,
    'toApp': {c: {'materialType': TRAIN_CLASS_TO_APP[c][0],
                  'grade': TRAIN_CLASS_TO_APP[c][1],
                  'note': TRAIN_CLASS_TO_APP[c][2]} for c in CLASSES},
    'unreachableMaterialTypes': UNREACHABLE_MATERIAL_TYPES,
    'reachableGrades': REACHABLE_GRADES,
}, indent=2, ensure_ascii=False))
size_kb = len(tfl) / 1024

itp = tf.lite.Interpreter(model_content=tfl); itp.allocate_tensors()
di, do = itp.get_input_details()[0], itp.get_output_details()[0]
z = np.zeros(di['shape'], dtype=di['dtype'])
for _ in range(3):
    itp.set_tensor(di['index'], z); itp.invoke()
t0 = time.time()
for _ in range(20):
    itp.set_tensor(di['index'], z); itp.invoke()
lat = (time.time() - t0) / 20 * 1000
shutil.rmtree(sm, ignore_errors=True)
print(f'model_int8.tflite  {size_kb:.0f} KB  |  {lat:.1f} ms per citra (CPU notebook ini)')
print('Ukur ulang latensi di perangkat Android target sebelum angkanya dikutip.')

In [ ]:
# @title Simpan metrik dan laporan
metrics = {
    'dataset': {'images': len(man), 'classes': CLASSES,
                'per_material': {m: counts.get(m, 0) for m in CLASSES},
                'per_source': dict(Counter(r['source'] for r in man)),
                'clusters': len({r['cluster'] for r in man}),
                'split': {'train': len(tr_i), 'val': len(va_i), 'test': len(te_i),
                          'policy': 'per klaster near-duplicate, 70/13/17 (Thung & Yang 2016)'},
                'sources': {s: SOURCE_META[s] for s in sorted({r['source'] for r in man})
                            if s in SOURCE_META}},
    'model': {'backbone': 'MobileNetV3-Small', 'input': [IMG, IMG, 3],
              'epochs_head': EPOCHS_HEAD, 'epochs_finetune': EPOCHS_FT},
    'test': {'accuracy': float((pred == y_te).mean()), 'macro_f1': macro_f1(y_te, pred, N),
             'per_class': {CLASSES[c]: v for c, v in per_class_prf(y_te, pred, N).items()}},
    'calibration': {'temperature': best_t, 'ece_before': e_before, 'ece_after': e_after},
    'abstain': {'threshold': TH, 'coverage': cov[pick], 'macro_f1_at_threshold': f1s[pick]},
    'cross_dataset': {f'{a}->{b}': v for (a, b), v in res.items()},
    'edge': {'tflite_int8_kb': size_kb, 'latency_ms_notebook_cpu': lat},
    'coverage_note': {'grades_reachable': REACHABLE_GRADES, 'grades_total': len(ALL_GRADES),
                      'material_types_unreachable': UNREACHABLE_MATERIAL_TYPES},
    'app_mapping': {c: {'materialType': TRAIN_CLASS_TO_APP[c][0],
                        'grade': TRAIN_CLASS_TO_APP[c][1]} for c in CLASSES},
}
(OUT / 'metrics.json').write_text(json.dumps(metrics, indent=2, ensure_ascii=False))

L = [f'# Hasil pelatihan TrashScan', '',
     f"{len(man):,} citra, {N} kelas material, dari "
     f"{len(set(r['source'] for r in man))} dataset publik.",
     f"Split per klaster near-duplicate ({metrics['dataset']['clusters']:,} klaster), 70/13/17.", '',
     '| Metrik | Nilai |', '|---|---|',
     f"| Akurasi (test) | {metrics['test']['accuracy']:.3f} |",
     f"| Macro-F1 (test) | {metrics['test']['macro_f1']:.3f} |",
     f"| ECE sebelum kalibrasi | {e_before:.3f} |",
     f"| ECE sesudah (T={best_t:.2f}) | {e_after:.3f} |",
     f"| Ambang abstain | {TH:.2f} (cakupan {cov[pick]:.0%}) |",
     f"| Model int8 | {size_kb:.0f} KB |", '',
     '## Per kelas', '', '| Kelas | P | R | F1 | n |', '|---|---|---|---|---|']
for c in range(N):
    v = metrics['test']['per_class'][CLASSES[c]]
    L.append(f"| {MATERIAL_LABEL_ID[CLASSES[c]]} | {v['precision']:.3f} | {v['recall']:.3f} "
             f"| {v['f1']:.3f} | {v['support']} |")
if res:
    L += ['', '## Lintas-dataset (macro-F1)', '', '| Dilatih | Diuji | Macro-F1 |', '|---|---|---|']
    L += [f'| {a} | {b} | {v:.3f} |' for (a, b), v in sorted(res.items())]
L += ['', '## Pemetaan ke enum aplikasi', '',
      'Kelas yang dilatih bukan nilai MaterialType. Inilah terjemahannya, dan '
      'inilah yang membuat model dapat dipasang sama sekali:', '',
      app_mapping_table(), '',
      f'MaterialType yang TIDAK dapat dicapai dari data publik: '
      f'{", ".join(UNREACHABLE_MATERIAL_TYPES)}. Keempatnya menuntut foto sendiri.']
L += ['', '## Batas yang wajib dinyatakan', '',
      f'Model ini mengeluarkan {N} kelas material. Papan harga BinGo bekerja pada '
      f'{len(ALL_GRADES)} grade, dan hanya {len(REACHABLE_GRADES)} di antaranya '
      f'({", ".join(REACHABLE_GRADES)}) yang dapat diturunkan dari dataset publik tanpa '
      'menebak. Pembedaan yang menentukan selisih harga terbesar — bening versus berwarna, '
      'koran versus duplex, tembaga versus besi — tidak dilabeli dataset mana pun.', '',
      'Model ini karena itu asisten identifikasi kasar, bukan penentu harga. '
      'Kode resin tetap jalur utama karena ia fakta, bukan tebakan.', '']
(OUT / 'report.md').write_text('\n'.join(L))
print('\n'.join(L[:16]))
print(f'\nArtefak: {OUT.resolve()}')

## Yang boleh dan tidak boleh diklaim

**Boleh** — angka macro-F1 dan per kelas dari test set milikmu sendiri; angka
lintas-dataset; ukuran model dan ambang abstain; perbandingan dengan Thung &
Yang karena rasio splitnya sama.

**Tidak boleh** — akurasi 97% dari makalah orang lain; "AI menentukan harga"
(harga datang dari bukti timbang, bukan dari model); latensi notebook sebagai
latensi perangkat; klaim grade apa pun di luar tiga yang tercapai.

## Langkah berikutnya

Bagian yang benar-benar baru bukan model ini, melainkan dataset yang belum ada:
foto material per-grade sebagaimana dipakai lapak Indonesia. 30–50 foto per
grade sudah cukup untuk mulai, dan pengumpulannya bisa digabung dengan wawancara
pemulung yang memang diminta juri. `label_map.py` sudah menyediakan lapis grade
yang tinggal diisi.

Untuk memasang model ke aplikasi: `react-native-fast-tflite` punya config plugin
Expo, butuh development build (bukan Expo Go), dan `visualClassifier.ts` sudah
memegang ambang abstain yang tinggal diganti dengan angka dari langkah 6.